In [ ]:
# Install required packages if needed
import subprocess
import sys
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

def install_package(pip_name, import_name=None, force_reinstall=False):
    """Install package using pip_name, check import using import_name or pip_name"""
    check_name = import_name if import_name else pip_name
    
    # Special handling for NumPy - check version first
    if pip_name.startswith('numpy'):
        try:
            import numpy
            if numpy.__version__.startswith('2.'):
                print(f"⚠ NumPy 2.x detected ({numpy.__version__}), downgrading to 1.x...")
                force_reinstall = True
            else:
                print(f"✓ NumPy {numpy.__version__} already installed (compatible)")
                return True
        except ImportError:
            pass
    
    if not force_reinstall:
        try:
            __import__(check_name)
            print(f"✓ {check_name} already installed")
            return True
        except ImportError:
            pass
    
    print(f"Installing {pip_name}...")
    try:
        cmd = [sys.executable, "-m", "pip", "install", pip_name]
        if force_reinstall:
            cmd.append("--force-reinstall")
        subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"✓ {pip_name} installed")
        return True
    except Exception as e:
        print(f"⚠ Failed to install {pip_name}: {e}")
        return False

# Install required packages (order matters - install numpy first if downgrading)
print("Checking and installing required packages...\n")

# First, handle NumPy separately to avoid compatibility issues
install_package('numpy<2.0', 'numpy', force_reinstall=False)

# Then install other packages
packages = [
    ('pyyaml', 'yaml'),  # pip install pyyaml, import yaml
    ('rclpy', 'rclpy'),
    ('cv-bridge', 'cv_bridge'),
    ('opencv-python', 'cv2'),
    ('matplotlib', 'matplotlib')
]

for pip_name, import_name in packages:
    install_package(pip_name, import_name)

print("\n✓ Package installation complete!")
print("\n⚠ IMPORTANT: If kernel crashes, restart kernel and run this cell again.")
print("Note: Some ROS 2 packages may need to be installed via apt:")
print("  sudo apt install ros-jazzy-cv-bridge ros-jazzy-sensor-msgs")

Checking and installing required packages...

⚠ NumPy 2.x detected (2.2.6), downgrading to 1.x...
Installing numpy<2.0...
✓ numpy<2.0 installed
✓ yaml already installed
✓ rclpy already installed
✓ cv_bridge already installed
✓ cv2 already installed
Installing matplotlib...
✓ matplotlib installed

✓ Package installation complete!

⚠ IMPORTANT: If kernel crashes, restart kernel and run this cell again.
Note: Some ROS 2 packages may need to be installed via apt:
  sudo apt install ros-jazzy-cv-bridge ros-jazzy-sensor-msgs


# RealSense Depth Camera Viewer

This notebook tests and displays the depth camera feed from RealSense D435.

## Prerequisites
1. RealSense camera driver must be running:
   ```bash
   ros2 launch realsense2_camera rs_launch.py
   ```
2. Make sure ROS 2 topics are available

## ⚠️ If Kernel Crashes

If the kernel crashes (especially on Cell 0 or Cell 2):
1. **Restart the kernel** (Kernel → Restart Kernel)
2. **Run Cell 0 first** - it will downgrade NumPy to fix compatibility
3. **Restart kernel again** after Cell 0 completes
4. **Then run cells in order**

**Alternative:** Use the standalone Python script instead:
```bash
cd ~/Documents/Robotic\ AI/Robotic-AI/piper/cubeAndLineDet
python3 test_depth_view.py
```

In [ ]:
# Import required libraries with error handling
try:
    import numpy as np
    print(f"✓ NumPy {np.__version__} imported")
    if np.__version__.startswith('2.'):
        print("⚠ WARNING: NumPy 2.x may cause compatibility issues!")
        print("  Consider downgrading: pip install 'numpy<2.0' --force-reinstall")
except ImportError as e:
    print(f"✗ Failed to import NumPy: {e}")
    raise

try:
    import rclpy
    from rclpy.node import Node
    from sensor_msgs.msg import Image, CameraInfo
    from cv_bridge import CvBridge
    print("✓ ROS 2 libraries imported")
except ImportError as e:
    print(f"✗ Failed to import ROS 2 libraries: {e}")
    print("  Install with: pip install rclpy")
    raise

try:
    import cv2
    print(f"✓ OpenCV {cv2.__version__} imported")
except ImportError as e:
    print(f"✗ Failed to import OpenCV: {e}")
    print("  Install with: pip install opencv-python")
    raise

try:
    import matplotlib.pyplot as plt
    print("✓ Matplotlib imported")
except ImportError as e:
    print(f"✗ Failed to import Matplotlib: {e}")
    print("  Install with: pip install matplotlib")
    raise

import time
from IPython.display import display, clear_output

print("\n✓ All imports successful!")

✓ NumPy 2.2.6 imported
⚠ WARNING: NumPy 2.x may cause compatibility issues!
  Consider downgrading: pip install 'numpy<2.0' --force-reinstall
✓ ROS 2 libraries imported
✓ OpenCV 4.12.0 imported
✗ Failed to import Matplotlib: cannot import name 'broadcast_to' from 'numpy.lib.stride_tricks' (/home/robotics_urop/Documents/Robotic AI/Robotic-AI/venv/lib/python3.12/site-packages/numpy/lib/stride_tricks.py)
  Install with: pip install matplotlib


ImportError: cannot import name 'broadcast_to' from 'numpy.lib.stride_tricks' (/home/robotics_urop/Documents/Robotic AI/Robotic-AI/venv/lib/python3.12/site-packages/numpy/lib/stride_tricks.py)

In [ ]:
# Initialize ROS 2 with error handling
try:
    if not rclpy.ok():
        rclpy.init()
    print("✓ ROS 2 initialized")
except Exception as e:
    print(f"✗ Failed to initialize ROS 2: {e}")
    print("  Make sure ROS 2 is properly installed")
    raise

# Check available topics
import subprocess
import os

print("\nChecking available ROS 2 topics...")
try:
    env = os.environ.copy()
    result = subprocess.run(
        ['ros2', 'topic', 'list'],
        capture_output=True,
        text=True,
        env=env,
        timeout=5
    )
    topics = result.stdout
    print("Available topics:")
    depth_topics = [t for t in topics.split('\n') if 'depth' in t.lower()]
    color_topics = [t for t in topics.split('\n') if 'color' in t.lower() and 'image' in t.lower()]
    
    if depth_topics:
        print("  Depth topics found:")
        for topic in depth_topics[:5]:
            print(f"    - {topic.strip()}")
    else:
        print("  ⚠ No depth topics found")
    
    if color_topics:
        print("  Color topics found:")
        for topic in color_topics[:5]:
            print(f"    - {topic.strip()}")
    else:
        print("  ⚠ No color topics found")
        
    if not depth_topics and not color_topics:
        print("\n⚠ Camera may not be running!")
        print("  Start camera: ros2 launch realsense2_camera rs_launch.py")
        
except Exception as e:
    print(f"⚠ Error checking topics: {e}")
    print("  Make sure ROS 2 is sourced: source /opt/ros/jazzy/setup.bash")

✓ ROS 2 initialized

Checking available ROS 2 topics...
Available topics:
  Depth topics found:
    - /camera/camera/depth/camera_info
    - /camera/camera/depth/image_rect_raw
    - /camera/camera/depth/metadata
    - /camera/camera/extrinsics/depth_to_color
    - /camera/camera/extrinsics/depth_to_depth
  Color topics found:
    - /camera/camera/color/image_raw


In [ ]:
class DepthViewer(Node):
    def __init__(self, depth_topic='/camera/camera/depth/image_rect_raw', 
                 color_topic='/camera/camera/color/image_raw',
                 node_name='depth_viewer'):
        super().__init__(node_name)
        
        self.bridge = CvBridge()
        self.depth_image = None
        self.color_image = None
        self.depth_received = False
        self.color_received = False
        self.depth_topic = depth_topic
        self.color_topic = color_topic
        
        # Subscribers
        self.depth_sub = self.create_subscription(
            Image,
            depth_topic,
            self.depth_callback,
            10
        )
        
        self.color_sub = self.create_subscription(
            Image,
            color_topic,
            self.color_callback,
            10
        )
        
        self.get_logger().info(f'Subscribed to depth topic: {depth_topic}')
        self.get_logger().info(f'Subscribed to color topic: {color_topic}')
    
    def depth_callback(self, msg):
        try:
            # Convert ROS Image to OpenCV format (16UC1 = 16-bit unsigned, single channel)
            self.depth_image = self.bridge.imgmsg_to_cv2(msg, desired_encoding='16UC1')
            self.depth_received = True
        except Exception as e:
            self.get_logger().error(f'Error processing depth image: {e}')
    
    def color_callback(self, msg):
        try:
            # Convert ROS Image to OpenCV format
            self.color_image = self.bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
            self.color_received = True
        except Exception as e:
            self.get_logger().error(f'Error processing color image: {e}')
    
    def get_depth_visualization(self):
        """Convert depth image to colorized visualization"""
        if self.depth_image is None:
            return None
        
        # Normalize depth to 0-255 (assuming max depth ~8m = 8000mm)
        depth_normalized = cv2.normalize(
            self.depth_image.astype(np.float32),
            None,
            0, 255,
            cv2.NORM_MINMAX
        ).astype(np.uint8)
        
        # Apply colormap (JET = blue=far, red=close)
        depth_colored = cv2.applyColorMap(depth_normalized, cv2.COLORMAP_JET)
        
        return depth_colored

print("✓ DepthViewer class defined")

✓ DepthViewer class defined


In [ ]:
# Quick check: Verify camera topics exist before subscribing
print("Quick check: Verifying camera topics exist...")
try:
    result = subprocess.run(
        ['ros2', 'topic', 'list'],
        capture_output=True,
        text=True,
        timeout=3
    )
    all_topics = result.stdout
    
    depth_topic = '/camera/camera/depth/image_rect_raw'
    color_topic = '/camera/camera/color/image_raw'
    
    depth_exists = depth_topic in all_topics
    color_exists = color_topic in all_topics
    
    if depth_exists:
        print(f"✓ Depth topic found: {depth_topic}")
    else:
        print(f"⚠ Depth topic NOT found: {depth_topic}")
        print("  Available depth topics:")
        for t in all_topics.split('\n'):
            if 'depth' in t.lower():
                print(f"    - {t.strip()}")
    
    if color_exists:
        print(f"✓ Color topic found: {color_topic}")
    else:
        print(f"⚠ Color topic NOT found: {color_topic}")
        print("  Available color topics:")
        for t in all_topics.split('\n'):
            if 'color' in t.lower() and 'image' in t.lower():
                print(f"    - {t.strip()}")
    
    if not depth_exists or not color_exists:
        print("\n⚠ Camera may not be running or topics are different!")
        print("  Start camera: ros2 launch realsense2_camera rs_launch.py")
        print("  Or check actual topics: ros2 topic list")
        
except Exception as e:
    print(f"⚠ Could not check topics: {e}")
    print("  Make sure ROS 2 is available")

Quick check: Verifying camera topics exist...
✓ Depth topic found: /camera/camera/depth/image_rect_raw
✓ Color topic found: /camera/camera/color/image_raw


: 

In [ ]:
# Create node with unique name
import uuid
node_id = str(uuid.uuid4())[:8]
node = DepthViewer(node_name=f'depth_viewer_{node_id}')

# Spin for a few seconds to receive data
print(f"Waiting for depth data from {node.depth_topic} (max 15 seconds)...")
print("Make sure camera is running: ros2 launch realsense2_camera rs_launch.py")
start_time = time.time()
timeout = 15.0
last_dot_time = start_time

while time.time() - start_time < timeout:
    rclpy.spin_once(node, timeout_sec=0.1)
    if node.depth_received:
        print(f"\n✓ Depth image received! Shape: {node.depth_image.shape}")
        print(f"  Data type: {node.depth_image.dtype}")
        print(f"  Min value: {node.depth_image.min()} mm")
        print(f"  Max value: {node.depth_image.max()} mm")
        break
    
    # Print progress dots every 0.5 seconds
    if time.time() - last_dot_time > 0.5:
        print(".", end="", flush=True)
        last_dot_time = time.time()

if not node.depth_received:
    print("\n⚠ No depth data received!")
    print("\nTroubleshooting:")
    print("  1. Check if camera is running:")
    print("     ros2 node list | grep camera")
    print("  2. Check available topics:")
    print("     ros2 topic list | grep depth")
    print("  3. Try starting camera:")
    print("     ros2 launch realsense2_camera rs_launch.py")
    print(f"  4. Current topic being subscribed: {node.depth_topic}")

Waiting for depth data from /camera/camera/depth/image_rect_raw (max 15 seconds)...
Make sure camera is running: ros2 launch realsense2_camera rs_launch.py


In [ ]:
# Display depth image if received
if node.depth_received and node.depth_image is not None:
    # Get visualization
    depth_vis = node.get_depth_visualization()
    
    # Create figure with subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Original depth (grayscale)
    axes[0].imshow(node.depth_image, cmap='gray')
    axes[0].set_title(f'Raw Depth Image\nShape: {node.depth_image.shape}\nType: {node.depth_image.dtype}')
    axes[0].axis('off')
    
    # Colorized depth
    axes[1].imshow(cv2.cvtColor(depth_vis, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Colorized Depth (JET colormap)\nBlue=Far, Red=Close')
    axes[1].axis('off')
    
    # Depth statistics
    valid_depth = node.depth_image[node.depth_image > 0]
    if len(valid_depth) > 0:
        axes[2].hist(valid_depth.flatten(), bins=50, edgecolor='black')
        axes[2].set_title(f'Depth Distribution\nMin: {valid_depth.min()}mm\nMax: {valid_depth.max()}mm\nMean: {valid_depth.mean():.1f}mm')
        axes[2].set_xlabel('Depth (mm)')
        axes[2].set_ylabel('Pixel Count')
    else:
        axes[2].text(0.5, 0.5, 'No valid depth data', ha='center', va='center')
        axes[2].set_title('Depth Distribution')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nDepth Image Statistics:")
    print(f"  Shape: {node.depth_image.shape}")
    print(f"  Data type: {node.depth_image.dtype}")
    print(f"  Min value: {node.depth_image.min()} mm")
    print(f"  Max value: {node.depth_image.max()} mm")
    print(f"  Valid pixels: {np.count_nonzero(node.depth_image)} / {node.depth_image.size}")
    if len(valid_depth) > 0:
        print(f"  Mean depth: {valid_depth.mean():.1f} mm")
        print(f"  Median depth: {np.median(valid_depth):.1f} mm")
else:
    print("⚠ No depth image to display")

⚠ No depth image to display


## Live View - Side by Side Color and Depth

In [ ]:
# Wait for both color and depth (use unique node name to avoid conflicts)
node_id2 = str(uuid.uuid4())[:8]
node2 = DepthViewer(node_name=f'depth_viewer_{node_id2}')

print("Waiting for color and depth images (max 15 seconds)...")
start_time = time.time()
timeout = 15.0
last_dot_time = start_time

while time.time() - start_time < timeout:
    rclpy.spin_once(node2, timeout_sec=0.1)
    if node2.depth_received and node2.color_received:
        print(f"\n✓ Both images received!")
        break
    
    # Print progress dots every 0.5 seconds
    if time.time() - last_dot_time > 0.5:
        print(".", end="", flush=True)
        last_dot_time = time.time()

if node2.depth_received and node2.color_received:
    depth_vis = node2.get_depth_visualization()
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Color image
    axes[0].imshow(cv2.cvtColor(node2.color_image, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Color Image\nShape: {node2.color_image.shape}')
    axes[0].axis('off')
    
    # Depth visualization
    axes[1].imshow(cv2.cvtColor(depth_vis, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Depth Visualization')
    axes[1].axis('off')
    
    # Overlay
    if node2.color_image.shape[:2] != depth_vis.shape[:2]:
        depth_resized = cv2.resize(depth_vis, (node2.color_image.shape[1], node2.color_image.shape[0]))
    else:
        depth_resized = depth_vis
    
    overlay = cv2.addWeighted(node2.color_image, 0.6, depth_resized, 0.4, 0)
    axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axes[2].set_title('Color + Depth Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Images displayed successfully!")
else:
    print("\n⚠ Did not receive both images")
    if not node2.depth_received:
        print(f"  - Depth image missing (topic: {node2.depth_topic})")
    if not node2.color_received:
        print(f"  - Color image missing (topic: {node2.color_topic})")
    print("\nMake sure camera is running: ros2 launch realsense2_camera rs_launch.py")

node2.destroy_node()

Waiting for color and depth images...
..

[WARN] [1768578372.025180780] [rcl.logging_rosout]: Publisher already registered for node name: 'depth_viewer'. If this is due to multiple nodes with the same name then all logs for the logger named 'depth_viewer' will go out over the existing publisher. As soon as any node with that name is destructed it will unregister the publisher, preventing any further logs for that name from being published on the rosout topic.
[INFO] [1768578372.032215071] [depth_viewer]: Subscribed to depth topic: /camera/camera/depth/image_rect_raw
[INFO] [1768578372.033809136] [depth_viewer]: Subscribed to color topic: /camera/camera/color/image_raw


..................................................................................................
⚠ Did not receive both images
  - Depth image missing
  - Color image missing


In [ ]:
# Cleanup all nodes
try:
    node.destroy_node()
except:
    pass

try:
    node2.destroy_node()
except:
    pass

print("✓ Cleanup complete")
print("\nNote: If you want to test again, make sure:")
print("  1. Camera is running: ros2 launch realsense2_camera rs_launch.py")
print("  2. Restart kernel and run cells from the beginning")

✓ Cleanup complete
